In [22]:
#pip install imageio-ffmpeg

   ---------------------------------------- 0.0/38.9 MB ? eta -:--:--
    --------------------------------------- 0.5/38.9 MB 9.8 MB/s eta 0:00:04
   - -------------------------------------- 1.6/38.9 MB 6.1 MB/s eta 0:00:07
   --- ------------------------------------ 3.7/38.9 MB 7.9 MB/s eta 0:00:05
   ----- ---------------------------------- 5.5/38.9 MB 8.2 MB/s eta 0:00:05
   ------- -------------------------------- 7.1/38.9 MB 7.8 MB/s eta 0:00:05
   --------- ------------------------------ 8.9/38.9 MB 8.3 MB/s eta 0:00:04
   ----------- ---------------------------- 10.7/38.9 MB 8.2 MB/s eta 0:00:04
   ------------- -------------------------- 12.8/38.9 MB 8.4 MB/s eta 0:00:04
   --------------- ------------------------ 14.7/38.9 MB 8.5 MB/s eta 0:00:03
   ---------------- ----------------------- 16.0/38.9 MB 8.3 MB/s eta 0:00:03
   ----------------- ---------------------- 17.0/38.9 MB 8.0 MB/s eta 0:00:03
   ------------------ --------------------- 18.1/38.9 MB 7.7 MB/s eta 0:00:03


<h2>3. Data preparation</h2>

Double check and make sure that all of our .wav files truly are converted correctly.

In [34]:
from pathlib import Path
import subprocess
import imageio_ffmpeg

SRC_DIR = Path("data/audio-samples")
TMP_DIR = Path("data/audio-wav")
FFMPEG_EXE = imageio_ffmpeg.get_ffmpeg_exe()

print("Start covertion to .wav file.")
for src_path in sorted(SRC_DIR.rglob("*.wav")):
    dst_path = (TMP_DIR / src_path.relative_to(SRC_DIR)).with_suffix(".wav")
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [FFMPEG_EXE, "-y", "-i", str(src_path), str(dst_path)],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=False,
    )
    print(f"Converting {src_path}...")

print("Convertion successful!")

Start covertion to .wav file.
Converting data\audio-samples\test\english\down\down-01.wav...
Converting data\audio-samples\test\english\down\down-02.wav...
Converting data\audio-samples\test\english\left\left-01.wav...
Converting data\audio-samples\test\english\left\left-02.wav...
Converting data\audio-samples\test\english\right\right-01.wav...
Converting data\audio-samples\test\english\right\right-02.wav...
Converting data\audio-samples\test\english\start\start-01.wav...
Converting data\audio-samples\test\english\start\start-02.wav...
Converting data\audio-samples\test\english\stop\stop-01.wav...
Converting data\audio-samples\test\english\stop\stop-02.wav...
Converting data\audio-samples\test\english\up\up-01.wav...
Converting data\audio-samples\test\english\up\up-02.wav...
Converting data\audio-samples\test\finnish\alas\alas-01.wav...
Converting data\audio-samples\test\finnish\alas\alas-02.wav...
Converting data\audio-samples\test\finnish\oikea\oikea-01.wav...
Converting data\audio-s

Convert our true .wav files to an image of 214x214 resolution for now. This resolution is really good for using with VGG16 model. This can be adjusted depending on the used pre-trained model.

In [31]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np

AUDIO_DIR = Path("data/audio-wav")
IMAGE_DIR = Path("data/image-samples")

IMG_SIZE = 224  # VGG16 deafult size (214x214)
DPI = 100
FIG_SIZE = IMG_SIZE / DPI

wav_files = sorted(AUDIO_DIR.rglob("*.wav"))
print(f"Found total of {len(wav_files)} wav files...\nBegin processing...")

for wav_path in wav_files:
    rel_path = wav_path.relative_to(AUDIO_DIR).with_suffix(".png")
    out_path = IMAGE_DIR / rel_path
    out_path.parent.mkdir(parents=True, exist_ok=True)
    
    y, sr = librosa.load(str(wav_path), sr=22050, mono=True)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    mel_db = librosa.power_to_db(mel, ref=np.max)

    fig, ax = plt.subplots(figsize=(FIG_SIZE, FIG_SIZE), dpi=DPI)
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
    ax.axis("off")

    librosa.display.specshow(
        mel_db,
        sr=sr,
        ax=ax,
        cmap="magma",
        vmin=-80,
        vmax=0,
    )

    plt.savefig(out_path, dpi=DPI)
    plt.close(fig)
    print(f"{out_path} ready...")

print("Processing completed! Images are succesfully saved in: data/image-samples.")

Found total of 144 wav files...
Begin processing...
data\image-samples\test\english\down\down-01.png ready...
data\image-samples\test\english\down\down-02.png ready...
data\image-samples\test\english\left\left-01.png ready...
data\image-samples\test\english\left\left-02.png ready...
data\image-samples\test\english\right\right-01.png ready...
data\image-samples\test\english\right\right-02.png ready...
data\image-samples\test\english\start\start-01.png ready...
data\image-samples\test\english\start\start-02.png ready...
data\image-samples\test\english\stop\stop-01.png ready...
data\image-samples\test\english\stop\stop-02.png ready...
data\image-samples\test\english\up\up-01.png ready...
data\image-samples\test\english\up\up-02.png ready...
data\image-samples\test\finnish\alas\alas-01.png ready...
data\image-samples\test\finnish\alas\alas-02.png ready...
data\image-samples\test\finnish\oikea\oikea-01.png ready...
data\image-samples\test\finnish\oikea\oikea-02.png ready...
data\image-sampl

Preparation done. Images are now ready to be used to train convolutional network.